# Cognitive Load Index (CLI) — Pipeline v1

**Project:** Predicting and Preventing Burnout Using Cognitive Load Analysis  
**Dataset:** Anders et al.  
**Stage:** Lab1 + Lab2 Features only (Wild excluded in v1)  
**Goal:** Binary classification Low vs High cognitive load → CLI score = P(High) × 100

---
### Label mapping
| Segment folder name | Cognitive load | Label |
|---|---|---|
| `relaxation_video`, `video_baseline` | Baseline / rest | 0 — Low |
| `*_easy` | Easy cognitive task | 0 — Low |
| `*_hard` | Hard cognitive task | 1 — High |
| anything else | Ambiguous | skipped |

### Features used (v1 — no EEG)
| File | Columns | Signal |
|---|---|---|
| `EDA_features.pickle` | `SCR_Peaks_N`, `SCR_Peaks_Amplitude_Mean` | Electrodermal activity |
| `HRV_features.pickle` | `HRV_MeanNN`, `HRV_SDNN`, `HRV_RMSSD`, `HRV_LFn`, `HRV_HFn`, `HRV_ratio_LFn_HFn` | Heart rate variability |
| `TEMP_features.pickle` | `mean_temp`, `std_temp` | Skin temperature |

In [ ]:
import pickle
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────
DATA_ROOT    = Path(".")
SESSIONS     = ["Lab1", "Lab2"]          # Wild excluded in v1
FEATURE_FILES = ["EDA_features", "HRV_features", "TEMP_features"]  # no EEG

# Segments to treat as Low (0) — exact name or suffix match
LOW_EXACT    = {"relaxation_video", "video_baseline"}
LOW_SUFFIX   = "_easy"
HIGH_SUFFIX  = "_hard"

In [ ]:
# ── Label assignment ─────────────────────────────────────────────────────────
def segment_to_label(segment_name: str):
    """
    Returns (label, load_level_str) or (None, None) if unclassifiable.
    """
    name = segment_name.lower()
    if name in LOW_EXACT or name.endswith(LOW_SUFFIX):
        return 0, "Low"
    if name.endswith(HIGH_SUFFIX):
        return 1, "High"
    return None, None

In [ ]:
# ── Robust data loader ───────────────────────────────────────────────────────
def load_segment(seg_dir: Path, pid: str, session: str):
    """
    Load one labeled segment from its Features subfolder.
    Returns (DataFrame, skip_reason_str).
    skip_reason_str is None on success, a message string on failure.
    """
    seg_name = seg_dir.name
    label, load_level = segment_to_label(seg_name)

    if label is None:
        return None, f"unrecognized segment name '{seg_name}'"

    dfs = []
    for feat_name in FEATURE_FILES:
        fpath = seg_dir / f"{feat_name}.pickle"
        if not fpath.exists():
            return None, f"{feat_name}.pickle missing"
        with open(fpath, "rb") as f:
            feat_df = pickle.load(f)
        if not isinstance(feat_df, pd.DataFrame):
            feat_df = pd.DataFrame(feat_df)
        dfs.append(feat_df.reset_index(drop=True))

    combined = pd.concat(dfs, axis=1)
    combined["label"]          = label
    combined["load_level"]     = load_level
    combined["participant_id"] = pid
    combined["session"]        = session
    combined["segment"]        = seg_name
    combined["window_idx"]     = range(len(combined))
    return combined, None


def load_all_features():
    """
    Walk all UN_* participant folders and load Lab1/Lab2 features.
    Missing sessions, missing Features folders, and missing files
    are all handled gracefully — each issue logs a warning and is
    recorded in the summary table.

    Returns
    -------
    dataset    : pd.DataFrame — all windows, stacked
    summary_df : pd.DataFrame — one row per participant
    """
    participant_dirs = sorted(DATA_ROOT.glob("UN_*"))
    if not participant_dirs:
        raise FileNotFoundError(f"No UN_* folders found under {DATA_ROOT.resolve()}")

    all_segments = []
    summary_rows = []

    for pdir in participant_dirs:
        if not pdir.is_dir():
            continue
        pid = pdir.name

        row = {
            "participant_id":   pid,
            "Lab1_used":        "no",
            "Lab2_used":        "no",
            "segments_loaded":  0,
            "segments_skipped": 0,
            "skip_reasons":     [],
        }

        for session in SESSIONS:
            session_path = pdir / session

            # ── Session folder missing ────────────────────────────────────
            if not session_path.exists():
                msg = f"{session} folder missing"
                warnings.warn(f"[{pid}] {msg} — skipping session")
                row["skip_reasons"].append(msg)
                continue

            # ── Features sub-folder missing ───────────────────────────────
            features_path = session_path / "Features"
            if not features_path.exists():
                msg = f"{session}/Features folder missing"
                warnings.warn(f"[{pid}] {msg} — skipping session")
                row["skip_reasons"].append(msg)
                continue

            seg_dirs = sorted(
                d for d in features_path.iterdir() if d.is_dir()
            )
            if not seg_dirs:
                msg = f"{session}/Features is empty"
                warnings.warn(f"[{pid}] {msg} — skipping session")
                row["skip_reasons"].append(msg)
                continue

            session_loaded = 0
            for seg_dir in seg_dirs:
                seg_df, reason = load_segment(seg_dir, pid, session)
                if reason is not None:
                    warnings.warn(
                        f"[{pid}/{session}/{seg_dir.name}] {reason} — skipping segment"
                    )
                    row["segments_skipped"] += 1
                    row["skip_reasons"].append(
                        f"{session}/{seg_dir.name}: {reason}"
                    )
                else:
                    all_segments.append(seg_df)
                    row["segments_loaded"] += 1
                    session_loaded += 1

            if session_loaded > 0:
                row[f"{session}_used"] = "yes"

        row["skip_reasons"] = (
            "; ".join(row["skip_reasons"]) if row["skip_reasons"] else ""
        )
        summary_rows.append(row)

    dataset = (
        pd.concat(all_segments, ignore_index=True)
        if all_segments
        else pd.DataFrame()
    )
    summary_df = pd.DataFrame(summary_rows)
    return dataset, summary_df

In [ ]:
# ── Run loader ───────────────────────────────────────────────────────────────
# Show all warnings as they happen (not just the first occurrence)
warnings.filterwarnings("always")

dataset, summary_df = load_all_features()

print(f"Dataset shape    : {dataset.shape}")
print(f"Total windows    : {len(dataset):,}")
print(f"Participants     : {dataset['participant_id'].nunique()}")
print(f"Sessions seen    : {sorted(dataset['session'].unique())}")
print(f"Feature columns  : {[c for c in dataset.columns if c not in ['label','load_level','participant_id','session','segment','window_idx']]}")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 90)
print("LOAD SUMMARY — one row per participant")
print("=" * 90)

# Widen display so skip_reasons column is readable
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

display(summary_df)

print()
total_loaded  = summary_df["segments_loaded"].sum()
total_skipped = summary_df["segments_skipped"].sum()
print(f"Total segments loaded  : {total_loaded}")
print(f"Total segments skipped : {total_skipped}")
print(f"Lab1 used by           : {(summary_df['Lab1_used'] == 'yes').sum()} participants")
print(f"Lab2 used by           : {(summary_df['Lab2_used'] == 'yes').sum()} participants")

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
print("Class distribution (windows):")
print(dataset["load_level"].value_counts().rename("windows").to_frame())
print()

print("Class distribution (segments):")
seg_counts = (
    dataset.drop_duplicates(subset=["participant_id", "session", "segment"])
           [["load_level"]]
           .value_counts()
           .rename("segments")
           .to_frame()
)
print(seg_counts)

In [ ]:
# ── Feature matrix ────────────────────────────────────────────────────────────
META_COLS = ["label", "load_level", "participant_id", "session", "segment", "window_idx"]
FEATURE_COLS = [c for c in dataset.columns if c not in META_COLS]

X = dataset[FEATURE_COLS]
y = dataset["label"]

print(f"Feature matrix: {X.shape}  ({len(FEATURE_COLS)} features × {len(X):,} windows)")
print()

nan_counts = X.isnull().sum()
print("NaN counts per feature (before fill):")
print(nan_counts)
print(f"Total NaN cells: {nan_counts.sum()}")
print()

# Fill NaNs with each feature's median.
# Affected rows: windows where HRV could not be computed (too few heartbeats)
# or EDA had no skin conductance response peaks.
X = X.fillna(X.median())
print(f"NaN cells after fill: {X.isnull().sum().sum()}")
print()
print("Feature statistics:")
display(X.describe().T)

In [ ]:
# ── Train / test split — by participant (no data leakage) ────────────────────
# We split by participant ID so no windows from the same person
# appear in both train and test sets.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = dataset["participant_id"]

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_pids = dataset.iloc[train_idx]["participant_id"].unique()
test_pids  = dataset.iloc[test_idx]["participant_id"].unique()

print(f"Train: {len(X_train):,} windows from {len(train_pids)} participants")
print(f"Test : {len(X_test):,} windows from {len(test_pids)} participants")
print(f"Test participants: {sorted(test_pids)}")

In [ ]:
# ── Train classifier ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]   # P(High)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low", "High"]))

print("Confusion Matrix:")
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual Low", "Actual High"],
    columns=["Pred Low", "Pred High"]
)
display(cm)

print(f"\nROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
importance = pd.Series(
    clf.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)
print("Feature importances:")
print(importance.to_string())

In [ ]:
# ── Compute CLI score ─────────────────────────────────────────────────────────
# CLI = P(High cognitive load) × 100  (0 = fully relaxed, 100 = max load)

dataset_test = dataset.iloc[test_idx].copy()
dataset_test["CLI_score"] = y_proba * 100

# Mean CLI per task type across all test windows
cli_by_segment = (
    dataset_test.groupby("segment")["CLI_score"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "CLI_mean", "std": "CLI_std", "count": "n_windows"})
    .sort_values("CLI_mean")
)

print("Mean CLI score per task type (test set):")
display(cli_by_segment.round(2))

In [ ]:
# ── Visualise CLI by task ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

colors = ["steelblue" if "easy" in s or s in {"relaxation_video","video_baseline"} else "tomato"
          for s in cli_by_segment.index]

cli_by_segment["CLI_mean"].plot.barh(ax=ax, color=colors, xerr=cli_by_segment["CLI_std"])
ax.set_xlabel("Mean CLI Score")
ax.set_title("Cognitive Load Index by Task Type\n(blue = Low label, red = High label)")
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()